# 02. Platform Adapters

Notebook 01 covered the gateway core: `SessionRouter`, the race-condition guard, and the
`AsyncioExecutor` that runs an `ArcAgent` in-process. This notebook drops down a layer.

**The adapter is the gateway's contract with the outside world.** Every chat platform you
want an Arc agent to live in — Slack, Telegram, Mattermost, a homegrown WebSocket UI, an
internal CLI, an SMS gateway — talks to the gateway through the *same* `BasePlatformAdapter`
Protocol. If you can implement four async methods, the gateway can drive your platform.

**You will learn:**
- Why arcgateway pushes platform specifics to the boundary instead of leaking them inward.
- The exact shape of `InboundEvent` — the normalised envelope every adapter must produce.
- How `DeliveryTarget` parses `platform:chat_id[:thread_id]` for outbound routing.
- The `Executor` Protocol the adapter dispatches into, and the `Delta` chunks it gets back.
- How `AsyncioExecutor` wires an `agent_factory` into the loop end-to-end.
- How to **author a custom adapter from scratch** — start, stop, emit, deliver.
- When `SubprocessExecutor` and `NATSExecutor` (T1.6 / multi-instance) are the right tier.
- The failure modes that come for free if you follow the contract — and what bites you if you don't.

Everything runs in-process against synthetic platforms. No bot tokens, no live HTTP, no
WebSocket servers — the contract is what we're learning, not any one wire format.


## 1. Setup


In [1]:
# Setup: make Arc packages importable from this notebook
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

_here = Path.cwd()
for _p in [_here, *_here.parents]:
    if (_p / "packages").is_dir() and (_p / "pyproject.toml").is_file():
        REPO_ROOT = _p
        break
else:
    raise RuntimeError("Could not locate Arc repo root")

# Add every package's src/ (and tests/, where present) to sys.path
for _pkg in (REPO_ROOT / "packages").iterdir():
    for _sub in ("src", "tests"):
        _path = _pkg / _sub
        if _path.is_dir() and str(_path) not in sys.path:
            sys.path.insert(0, str(_path))

load_dotenv(REPO_ROOT / ".env")

False

Confirm the package version. `__version__` bumps with public-API changes; if you fork the
notebook against a different version, expect drift.


In [2]:
import arcgateway

print("arcgateway version:", arcgateway.__version__)
print("public exports:", sorted(arcgateway.__all__))

arcgateway version: 0.2.0
public exports: ['AsyncioExecutor', 'DeliveryTarget', 'Delta', 'Executor', 'GatewayRunner', 'InboundEvent', 'SessionRouter', 'build_session_key']


## 2. Why platform adapters — abstraction at the boundary

Slack pushes events through Socket Mode WebSockets. Telegram is long-poll over HTTPS.
Discord uses gateway intents and shards. Mattermost is webhook-driven. WhatsApp is RPC over
a queue. **The gateway should not care.**

If the gateway core had even a single `if platform == "slack"` branch, every new platform
would mean editing a hot path. The whole point of a gateway is to be platform-shaped at the
edges and platform-agnostic everywhere else.

Arc's design decision (SDD §3.1):

1. **Adapters own the connection.** Long-poll loops, websocket reconnects, OAuth refresh —
   that's adapter business. The runner just calls `connect()` and `disconnect()`.
2. **Adapters normalise inbound events.** Whatever the platform delivers — JSON blob, gRPC
   message, raw bytes — the adapter wraps it as an `InboundEvent` and hands it to the
   `SessionRouter`.
3. **Adapters format outbound deltas.** The executor streams `Delta` chunks; the adapter
   decides how to render them on the platform (Slack message, Telegram edit, WebSocket frame).
4. **Adapters never see secrets they don't need.** The web adapter never sees the viewer's
   bearer token — the route resolves `user_did` and `chat_id` and passes them in.

Below is the layering, viewed from outside in:


In [3]:
from textwrap import dedent

diagram = dedent("""
    ┌───────────────────────────────────────────────────────────────┐
    │  PLATFORM (Slack, Telegram, web, SMS, ...)                    │
    │   ▲                                       │                   │
    │   │ outbound: adapter.send(target, msg)   ▼ inbound: HTTP/WS  │
    ├───┼───────────────────────────────────────┼───────────────────┤
    │   │            BasePlatformAdapter        │                   │
    │   │   (connect / disconnect / send / send_with_id)            │
    │   │                                       │                   │
    │   │      ┌────────────────────────┐       │                   │
    │   └──────┤  StreamBridge          │       │                   │
    │          │  (executor → adapter)  │       │                   │
    │          └─────────▲──────────────┘       ▼                   │
    │                    │             ┌───────────────────────┐    │
    │                    │             │  SessionRouter        │    │
    │            Delta stream          │  (race guard, FIFO)   │    │
    │                    │             └────────┬──────────────┘    │
    │            ┌───────┴───────┐              │ InboundEvent      │
    │            │  Executor     │ ◄────────────┘                   │
    │            │  (asyncio /   │                                  │
    │            │   subprocess) │                                  │
    │            └───────────────┘                                  │
    └───────────────────────────────────────────────────────────────┘
""").strip("\n")

print(diagram)

┌───────────────────────────────────────────────────────────────┐
│  PLATFORM (Slack, Telegram, web, SMS, ...)                    │
│   ▲                                       │                   │
│   │ outbound: adapter.send(target, msg)   ▼ inbound: HTTP/WS  │
├───┼───────────────────────────────────────┼───────────────────┤
│   │            BasePlatformAdapter        │                   │
│   │   (connect / disconnect / send / send_with_id)            │
│   │                                       │                   │
│   │      ┌────────────────────────┐       │                   │
│   └──────┤  StreamBridge          │       │                   │
│          │  (executor → adapter)  │       │                   │
│          └─────────▲──────────────┘       ▼                   │
│                    │             ┌───────────────────────┐    │
│                    │             │  SessionRouter        │    │
│            Delta stream          │  (race guard, FIFO)   │    │
│         

Read it as: **the adapter is the only layer that knows what platform we're on.** Everything
below the dashed line speaks Arc primitives — `InboundEvent`, `DeliveryTarget`, `Delta`.

That decision pays dividends in three places. **Testability:** you can drive the gateway
with synthetic events and never touch real wire formats. **Auditability:** the
`InboundEvent` is the single point where platform-specific data crosses into Arc — every
audit/policy decision keys off the same struct. **Replaceability:** swap Slack for
Mattermost without touching `SessionRouter` or any executor.


## 3. `InboundEvent` envelope — what every adapter must produce

Whatever shape the platform's payload takes, the adapter must hand the router an
`InboundEvent`. This is the *one* type the gateway's interior speaks. Everything else —
Slack `event` blobs, Telegram `Update` objects, custom JSON — stops at the adapter.


In [4]:
from arcgateway.executor import InboundEvent
import inspect

print(f"{'Field':<14} Type")
for name, field in InboundEvent.model_fields.items():
    print(f"  {name:<14} {field.annotation}")

Field          Type
  platform       <class 'str'>
  chat_id        <class 'str'>
  thread_id      str | None
  user_did       <class 'str'>
  agent_did      <class 'str'>
  session_key    <class 'str'>
  message        <class 'str'>
  raw_payload    dict[str, typing.Any]


The fields fall into three groups.

**Platform locator** — `platform`, `chat_id`, `thread_id`. "Where on the platform did this
happen?" The same triple shows up in `DeliveryTarget` so the response can come back to the
exact same surface.

**Identity** — `user_did`, `agent_did`. The adapter has already resolved the platform-native
user (`@joshs`, `slack U02ABC`, telegram `12345`) into Arc's cross-platform DID and chosen
which agent should handle the message. By the time `SessionRouter` sees the event, no
platform-specific identity decisions remain.

**Payload** — `message`, `session_key`, `raw_payload`. `message` is what the agent sees.
`session_key` is the deterministic `(agent_did, user_did)` digest from notebook 01. And
`raw_payload` is the original platform blob — kept for audit, replay, debugging — so you
can always trace back to what the platform actually delivered.


In [5]:
from arcgateway.executor import InboundEvent
from arcgateway.session import build_session_key

agent_did = "did:arc:agent:demo"
user_did = "did:arc:user:alice"

event = InboundEvent(
    platform="telegram",
    chat_id="12345",
    thread_id=None,
    user_did=user_did,
    agent_did=agent_did,
    session_key=build_session_key(agent_did, user_did),
    message="hello agent",
    raw_payload={"update_id": 42, "from": {"id": 12345, "username": "alice"}},
)

print("session_key:", event.session_key)
print("raw payload retained:", event.raw_payload)

session_key: 6cc4ad1d6db637d4
raw payload retained: {'update_id': 42, 'from': {'id': 12345, 'username': 'alice'}}


Two non-obvious rules.

**`raw_payload` defaults to `{}`, not `None`.** That means audit code can always do
`event.raw_payload.get("update_id")` without a None-check. Pydantic's `Field(default_factory=dict)`
guarantees a fresh dict per event — no aliased mutable default landmines.

**Validation runs at construction.** Drop a field and Pydantic raises immediately. This is
your *only* schema check — there's no separate "is this event valid?" step downstream.


In [6]:
from pydantic import ValidationError

try:
    InboundEvent(
        platform="slack",
        chat_id="C123",
        # user_did intentionally missing
        agent_did="did:arc:agent:demo",
        session_key="k",
        message="hi",
    )
except ValidationError as exc:
    print("ValidationError raised — required field missing:")
    print(exc.errors()[0])

ValidationError raised — required field missing:
{'type': 'missing', 'loc': ('user_did',), 'msg': 'Field required', 'input': {'platform': 'slack', 'chat_id': 'C123', 'agent_did': 'did:arc:agent:demo', 'session_key': 'k', 'message': 'hi'}, 'url': 'https://errors.pydantic.dev/2.12/v/missing'}


## 4. `DeliveryTarget` — parsing `platform:chat_id[:thread_id]`

Outbound is the inverse problem. Given an instruction like "reply to that Slack thread" or
"DM this Telegram user", we need a string-addressable handle that survives:

- **TOML config** — `deliver_to = "slack:C0123:T9876"` in a scheduled-job file.
- **CLI arguments** — `arc message-send slack:C0123 'hello'` with no quoting drama.
- **JSON payloads** — a single string field beats an embedded object every time.

Hence the colon-delimited format: `platform:chat_id` or `platform:chat_id:thread_id`.


In [7]:
from arcgateway.delivery import DeliveryTarget

for s in [
    "telegram:12345",
    "telegram:12345:67890",
    "slack:C0123ABC",
    "slack:C0123ABC:T9876",
    "web:viewer-uuid:tab-7",
]:
    target = DeliveryTarget.parse(s)
    print(
        f"{s!r:<35} → platform={target.platform!r:<12} chat_id={target.chat_id!r:<12} thread_id={target.thread_id!r}"
    )

'telegram:12345'                    → platform='telegram'   chat_id='12345'      thread_id=None
'telegram:12345:67890'              → platform='telegram'   chat_id='12345'      thread_id='67890'
'slack:C0123ABC'                    → platform='slack'      chat_id='C0123ABC'   thread_id=None
'slack:C0123ABC:T9876'              → platform='slack'      chat_id='C0123ABC'   thread_id='T9876'
'web:viewer-uuid:tab-7'             → platform='web'        chat_id='viewer-uuid' thread_id='tab-7'


**Round-trip.** `str(target) == original`. That property matters: the audit log stores the
string form, and the runtime should be able to round-trip it back into a struct without
loss.


In [8]:
import json

for s in ["telegram:12345", "slack:C0123ABC:T9876"]:
    target = DeliveryTarget.parse(s)
    print(f"original={s!r}  →  str(target)={str(target)!r}  →  match={str(target) == s}")

# JSON serialisation works because pydantic emits a flat object
target = DeliveryTarget.parse("slack:C0123ABC:T9876")
print("\nas JSON:", json.dumps(target.model_dump()))

original='telegram:12345'  →  str(target)='telegram:12345'  →  match=True
original='slack:C0123ABC:T9876'  →  str(target)='slack:C0123ABC:T9876'  →  match=True

as JSON: {"platform": "slack", "chat_id": "C0123ABC", "thread_id": "T9876"}


**Platform name is normalised.** `Telegram` → `telegram`, leading/trailing whitespace
stripped. So `deliver_to = "  Telegram:1234 "` from a hand-edited TOML file still works.
Unknown platforms (anything not in the known set) **don't raise** — adapters can be
registered at runtime, and the parser shouldn't predict the registry — they just log a
warning so misconfigurations surface during dev.


In [9]:
import logging
import io

buf = io.StringIO()
handler = logging.StreamHandler(buf)
handler.setLevel(logging.WARNING)
logging.getLogger("arcgateway.delivery").addHandler(handler)

# Unknown platforms log a warning but don't raise.
target = DeliveryTarget.parse("my-internal-cli:agent-007")
print("parsed unknown platform:", target.platform, target.chat_id)
print("warning emitted:", "my-internal-cli" in buf.getvalue())

parsed unknown platform: my-internal-cli agent-007
warning emitted: True


**Validation rejects empty `chat_id`.** Some quick sharp-edge cases that *do* raise:


In [10]:
from arcgateway.delivery import DeliveryTarget

for bad in ["telegram", "telegram:", ""]:
    try:
        DeliveryTarget.parse(bad)
    except ValueError as exc:
        print(f"{bad!r:<14} → ValueError: {exc}")

'telegram'     → ValueError: Invalid DeliveryTarget format 'telegram'. Expected 'platform:chat_id' or 'platform:chat_id:thread_id'.
'telegram:'    → ValueError: 1 validation error for DeliveryTarget
chat_id
  Value error, chat_id must not be empty [type=value_error, input_value='', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error
''             → ValueError: Invalid DeliveryTarget format ''. Expected 'platform:chat_id' or 'platform:chat_id:thread_id'.


## 5. `Executor` Protocol — what the adapter dispatches into

An adapter never invokes an LLM. It hands the inbound event to the `SessionRouter`, which
in turn dispatches to the *executor*. Multiple executors exist; the gateway's tier policy
picks one. **Every executor satisfies the same Protocol.**


In [11]:
from arcgateway.executor import Executor
import inspect

print("Executor is runtime-checkable:", hasattr(Executor, "__instancecheck__"))
print("\n--- Executor Protocol contract ---")
print(inspect.getdoc(Executor))

Executor is runtime-checkable: True

--- Executor Protocol contract ---
Contract for running an ArcAgent in response to an InboundEvent.

run() is a coroutine that returns an AsyncIterator[Delta]. It is NOT
an async generator function. This separation means:
- Callers can ``delta_iter = await executor.run(event)`` to set up context.
- Streaming happens when the caller does ``async for delta in delta_iter``.
- Failures before streaming begins (auth, connection) raise from run().

All implementations must:
- Be safe to call concurrently (multiple sessions in parallel).
- Never share mutable state across concurrent run() calls.
- Always yield a final Delta(kind="done", is_final=True) as the last item.
- Emit structured logs (never print statements).


Two things to internalise about the contract.

**`run()` is `async def` returning an `AsyncIterator[Delta]`.** It is *not* an async
generator function. Setup happens in `run()` itself — auth checks, rate limit reservations,
creating a session record. Streaming happens in the iterator it returns. Failures *before*
streaming begins (e.g., "agent not found") raise from `run()`; failures *during* streaming
yield error deltas. The split lets callers `delta_iter = await executor.run(event)` to
detect setup failures *without* starting consumption.

**Always end with a `done` sentinel.** Every implementation MUST yield a final
`Delta(kind="done", is_final=True)` — even on error. `StreamBridge` uses that signal to
release the session lock so the next message can run. Forget it and the session deadlocks.


Here's the `Delta` type — the streaming chunk every executor produces:


In [12]:
from arcgateway.executor import Delta

print("Delta fields:")
for name, field in Delta.model_fields.items():
    print(f"  {name:<10} {field.annotation}")

# A few examples of what executors yield in the wild:
examples = [
    Delta(kind="token", content="Hello ", is_final=False, turn_id="t1"),
    Delta(kind="token", content="world.", is_final=False, turn_id="t1"),
    Delta(kind="tool_call", content='search_web("arc agent")', is_final=False, turn_id="t1"),
    Delta(kind="done", content="", is_final=True, turn_id="t1"),
]
for d in examples:
    print(f"  kind={d.kind:<10} is_final={d.is_final} content={d.content!r}")

Delta fields:
  kind       typing.Literal['token', 'tool_call', 'done']
  content    <class 'str'>
  is_final   <class 'bool'>
  turn_id    <class 'str'>
  kind=token      is_final=False content='Hello '
  kind=token      is_final=False content='world.'
  kind=tool_call  is_final=False content='search_web("arc agent")'
  kind=done       is_final=True content=''


**`turn_id` is per-turn, not per-event.** A single inbound event might trigger several tool
calls before the LLM emits its final response — they all share the same `turn_id`. That's
the unit of idempotency: if the connection drops mid-turn and the bridge replays, it keys
on `turn_id` to dedupe.


## 6. `AsyncioExecutor` end-to-end

Notebook 01 introduced `AsyncioExecutor` and wired an `agent_factory` into it.
Here we exercise the full inbound-to-outbound path **without** a `SessionRouter` —
calling the executor directly so we can see exactly what an adapter would receive.


In [13]:
import asyncio

from arcgateway.executor import AsyncioExecutor, InboundEvent
from arcgateway.session import build_session_key

executor = AsyncioExecutor()  # echo stub — no agent_factory


async def echo_demo():
    event = InboundEvent(
        platform="demo",
        chat_id="demo-1",
        user_did="did:arc:user:alice",
        agent_did="did:arc:agent:bot",
        session_key=build_session_key("did:arc:agent:bot", "did:arc:user:alice"),
        message="ping",
    )
    deltas = []
    delta_iter = await executor.run(event)
    async for d in delta_iter:
        deltas.append(d)
    return deltas


deltas = await echo_demo()
for d in deltas:
    print(f"{d.kind:<6} is_final={d.is_final} turn={d.turn_id}: {d.content!r}")

token  is_final=False turn=ad0e9e91dfe7a640: "[AsyncioExecutor stub] Received: 'ping' (session=ad0e9e91dfe7a640)"
done   is_final=True turn=ad0e9e91dfe7a640: ''


Two non-final tokens? No — one token plus the `done` sentinel. The echo stub is the
minimum viable executor: it echoes the message, then closes the turn. Every real executor
follows the same contract; only the *content* of the deltas changes.

Now wire a real agent factory. `AsyncioExecutor`'s real contract (`arcgateway/executor.py`)
calls `session = await agent.session(event.session_key)` to open/resume the agent's
persistent session, then consumes `async for stream_event in agent.run(event.message,
session=session)` — an `arcrun` `StreamEvent` iterator, adapting each `TokenEvent` into a
token `Delta`. There is no `chat()` fallback — the agent must expose both `session()` and
`run()`.

In [14]:
from arcgateway.executor import AsyncioExecutor, InboundEvent
from arcgateway.session import build_session_key
from arcrun import TokenEvent


class ToyAgent:
    """Stand-in for ArcAgent. The real one lives in arcagent.

    Implements the real executor contract: `session()` opens/resumes a
    session handle, `run()` is an async generator yielding arcrun
    StreamEvents (TokenEvent here) — there is no `chat()` shortcut.
    """

    def __init__(self, agent_did: str) -> None:
        self.agent_did = agent_did
        self._memory: dict[str, list[str]] = {}

    async def session(self, session_key: str) -> str:
        # A real ArcAgent opens/resumes a persistent SessionManager log here.
        return session_key

    async def run(self, message: str, *, session: str):
        history = self._memory.setdefault(session, [])
        history.append(message)
        yield TokenEvent(text=f"Heard {len(history)} message(s) on {session[:8]}")


_agent = ToyAgent("did:arc:agent:bot")


async def factory(agent_did: str):
    return _agent  # cache one instance per (agent_did) in real code


executor = AsyncioExecutor(agent_factory=factory)
print("agent_factory wired:", executor.agent_factory is not None)

agent_factory wired: True


Same call, different deltas — the executor delegates to the agent's `chat()`:


In [15]:
async def factory_demo():
    session_key = build_session_key("did:arc:agent:bot", "did:arc:user:alice")
    outputs = []
    for msg in ["hello", "how are you", "goodbye"]:
        event = InboundEvent(
            platform="demo",
            chat_id="demo-1",
            user_did="did:arc:user:alice",
            agent_did="did:arc:agent:bot",
            session_key=session_key,
            message=msg,
        )
        delta_iter = await executor.run(event)
        async for d in delta_iter:
            if d.kind == "token":
                outputs.append(d.content)
    return outputs


for line in await factory_demo():
    print(line)

Heard 1 message(s) on ad0e9e91
Heard 2 message(s) on ad0e9e91
Heard 3 message(s) on ad0e9e91


Notice how the toy agent's history grows across calls — the executor passes the same
`session_key` as `session_id`, so the agent sees a continuous session no matter how many
platforms the user is messaging from.

**Errors become deltas, not exceptions.** If the agent's `chat()` raises, the executor
catches it, yields a `[agent-error] ...` token, and *still* yields the `done` sentinel. The
session lock releases; the user sees the error; the gateway lives.


In [16]:
class FlakyAgent:
    async def session(self, session_key: str) -> str:
        return session_key

    async def run(self, message: str, *, session: str):
        raise RuntimeError("upstream LLM timed out")
        yield  # pragma: no cover — makes this an async generator; never reached


_flaky = FlakyAgent()


async def flaky_factory(agent_did: str):
    return _flaky


flaky_executor = AsyncioExecutor(agent_factory=flaky_factory)


async def flaky_demo():
    event = InboundEvent(
        platform="demo",
        chat_id="demo-1",
        user_did="did:arc:user:alice",
        agent_did="did:arc:agent:bot",
        session_key="flaky-key-0001",
        message="hello",
    )
    deltas = []
    async for d in await flaky_executor.run(event):
        deltas.append(d)
    return deltas


for d in await flaky_demo():
    print(f"{d.kind:<6} is_final={d.is_final}: {d.content!r}")

AsyncioExecutor: agent error session=flaky-key-0001: upstream LLM timed out
Traceback (most recent call last):
  File "/Users/joshschultz/Projects/arc/.claude/worktrees/agent-af67a1128a112c184/packages/arcgateway/src/arcgateway/executor.py", line 256, in _stream
    async for stream_event in agent.run(event.message, session=session):
  File "/var/folders/tc/283s9y5x2ws8rtq0c9skdn700000gn/T/ipykernel_12293/1985495861.py", line 6, in run
    raise RuntimeError("upstream LLM timed out")
RuntimeError: upstream LLM timed out


token  is_final=False: '[agent-error] the run failed; see server logs'
done   is_final=True: ''


**`set_agent_factory` lets arcui's `embedded_agents` wrap the factory after construction.**
The wrapper composes a cache + fleet-registry hook around the underlying loader without
mutating private state. Skim the contract:


In [17]:
from arcgateway.executor import AsyncioExecutor

executor = AsyncioExecutor()


async def base_factory(agent_did):
    print(f"  base_factory called for {agent_did}")
    return ToyAgent(agent_did)


executor.set_agent_factory(base_factory)

# Wrap it: cache + log
_cache: dict = {}
underlying = executor.agent_factory


async def cached_factory(agent_did: str):
    if agent_did not in _cache:
        _cache[agent_did] = await underlying(agent_did)
    print(f"  cached_factory hit/{len(_cache)} for {agent_did}")
    return _cache[agent_did]


executor.set_agent_factory(cached_factory)


async def warmup():
    for _ in range(2):
        await executor.agent_factory("did:arc:agent:bot")


await warmup()

  base_factory called for did:arc:agent:bot
  cached_factory hit/1 for did:arc:agent:bot
  cached_factory hit/1 for did:arc:agent:bot


## 7. Authoring a custom adapter — synthetic Slack-like example

Now we do the thing the gateway is for: **stand up a brand-new platform adapter from
scratch.** No bot tokens, no real Slack — we'll simulate Slack's Socket Mode in-memory so
the lifecycle is observable end-to-end.

First, the contract. Every adapter must satisfy `BasePlatformAdapter`:


In [18]:
from arcgateway.adapters.base import BasePlatformAdapter
import inspect

print(inspect.getdoc(BasePlatformAdapter))

Protocol that all platform adapters must satisfy.

Adapters are responsible for:
1. Maintaining a connection to their platform (long-poll, WebSocket, etc.).
2. Emitting normalised InboundEvents to the SessionRouter.
3. Delivering formatted messages back to users via send().

Minimum surface — adapters implement exactly these three methods.
Additional capabilities (e.g., edit_message, delete_message, send_file)
are optional and NOT part of this Protocol to keep the base surface minimal.

The event-source loop (polling/websocket) runs as an asyncio.Task owned
by GatewayRunner and is NOT part of this Protocol — adapters yield events
from their own internal mechanism and call a registered callback instead.
GatewayRunner registers the callback via connect().


Four async methods plus a `name` attribute. That's the entire surface.

- **`name`** — string identifier matched by `DeliveryTarget.platform`. `"slack"`,
  `"telegram"`, `"my-internal-cli"`. Lowercase, stable.
- **`connect()`** — open the platform connection. Start any background tasks. Return
  promptly. *Don't block the gateway startup.*
- **`disconnect()`** — graceful shutdown. Cancel anything `connect()` started. **Never
  raise** — log and return.
- **`send(target, message, *, reply_to=None)`** — push a message to the platform. The
  outbound side; called by `StreamBridge`.
- **`send_with_id(target, message)`** — like `send` but returns the platform-assigned
  message ID. Default falls back to `send()` and returns `None`. Override only if your
  platform actually returns IDs (Slack does, Telegram does, plain webhooks usually don't).


### 7.1 The synthetic platform

Real Slack delivers events through a Socket Mode WebSocket. We'll model that as an
in-memory `asyncio.Queue` of payloads. Tests against this faux platform behave exactly
like the real adapter would — same async surface, same lifecycle.


In [19]:
import asyncio
from typing import Any


class FakeSlackPlatform:
    """Stands in for Slack's Socket Mode WebSocket.

    Real Slack: events arrive as JSON frames over a WebSocket; outbound
    calls go to chat.postMessage. We replace both with in-memory queues.
    """

    def __init__(self) -> None:
        self._inbox: asyncio.Queue[dict] = asyncio.Queue()
        self.outbox: list[dict] = []
        self._next_msg_id = 100

    async def push_user_message(self, *, user_id: str, channel: str, text: str) -> None:
        """User-side helper: pretend a Slack user typed something."""
        await self._inbox.put(
            {
                "type": "event_callback",
                "event": {"type": "message", "user": user_id, "channel": channel, "text": text},
            }
        )

    async def next_event(self) -> dict:
        """Adapter-side: block until next inbound event."""
        return await self._inbox.get()

    async def post_message(self, channel: str, text: str) -> str:
        """Adapter-side: 'POST chat.postMessage' equivalent."""
        msg_id = f"msg_{self._next_msg_id}"
        self._next_msg_id += 1
        self.outbox.append({"id": msg_id, "channel": channel, "text": text})
        return msg_id

### 7.2 The adapter itself

Now build `SyntheticSlackAdapter` against `BasePlatformAdapter`. Read it carefully — every
real adapter (`telegram.py`, `slack.py`, `web.py`) follows the same shape.


In [20]:
import logging
from collections.abc import Awaitable, Callable

from arcgateway.adapters.base import BasePlatformAdapter
from arcgateway.delivery import DeliveryTarget
from arcgateway.executor import InboundEvent
from arcgateway.session import build_session_key

_log = logging.getLogger("adapter.synthetic_slack")

OnInbound = Callable[[InboundEvent], Awaitable[None]]


class SyntheticSlackAdapter:
    """Slack-shaped adapter against an in-memory platform."""

    name = "slack"  # MUST match DeliveryTarget.platform for this platform

    def __init__(
        self,
        platform: FakeSlackPlatform,
        *,
        agent_did: str,
        allowed_user_ids: set[str],
        on_message: OnInbound,
    ) -> None:
        self._platform = platform
        self._agent_did = agent_did
        self._allowed = set(allowed_user_ids)
        self._on_message = on_message
        self._poll_task: asyncio.Task | None = None
        self._stopping = False

    # --- lifecycle ---

    async def connect(self) -> None:
        if self._poll_task is not None:
            return  # already connected
        self._stopping = False
        self._poll_task = asyncio.create_task(self._poll_loop(), name="slack-poll")
        _log.info("SyntheticSlackAdapter connected (agent=%s)", self._agent_did)

    async def disconnect(self) -> None:
        self._stopping = True
        if self._poll_task is not None:
            self._poll_task.cancel()
            try:
                await self._poll_task
            except (asyncio.CancelledError, Exception):
                pass  # disconnect must not raise
            self._poll_task = None
        _log.info("SyntheticSlackAdapter disconnected")

    # --- inbound ---

    async def _poll_loop(self) -> None:
        while not self._stopping:
            try:
                payload = await self._platform.next_event()
            except asyncio.CancelledError:
                return
            except Exception as exc:
                _log.exception("poll loop error: %s", exc)
                await asyncio.sleep(0.5)
                continue
            await self._handle_payload(payload)

    async def _handle_payload(self, payload: dict) -> None:
        event = payload.get("event") or {}
        if event.get("type") != "message":
            return  # filter at the adapter, before the router sees it
        user = event.get("user")
        if not user or user not in self._allowed:
            _log.warning("dropping unauthorised slack user %r", user)
            return
        user_did = f"did:arc:user:slack:{user}"
        inbound = InboundEvent(
            platform=self.name,
            chat_id=event["channel"],
            thread_id=event.get("thread_ts"),
            user_did=user_did,
            agent_did=self._agent_did,
            session_key=build_session_key(self._agent_did, user_did),
            message=event.get("text", ""),
            raw_payload=payload,
        )
        await self._on_message(inbound)

    # --- outbound ---

    async def send(
        self,
        target: DeliveryTarget,
        message: str,
        *,
        reply_to: str | None = None,
    ) -> None:
        if target.platform != self.name:
            msg = f"target.platform={target.platform!r} but adapter.name={self.name!r}"
            raise ValueError(msg)
        await self._platform.post_message(target.chat_id, message)

    async def send_with_id(self, target: DeliveryTarget, message: str) -> str | None:
        if target.platform != self.name:
            return None
        return await self._platform.post_message(target.chat_id, message)

Things worth pointing out about the implementation.

**Filtering happens at the adapter, before the router.** The `event.get("type") != "message"`
check, the allowlist check — both happen *here*. The router never sees the noise. That's
the whole point of normalising at the boundary.

**`connect()` returns promptly.** It launches a background task; it does *not* block waiting
for traffic. Otherwise gateway startup would stall on the first slow platform.

**`disconnect()` is robust.** Cancellation, exception swallowing, idempotent (calling it
twice is fine). Real adapters log richer diagnostics here, but the contract is: never raise.

**`raw_payload` carries the original event.** If anything goes sideways downstream, audit
code can recover the platform's exact bytes.


### 7.3 Wiring it into a SessionRouter

The adapter on its own doesn't run agents — it hands events to a callback. The callback is
almost always `SessionRouter.handle`. Let's hook the synthetic adapter to a router and run
messages through.


In [21]:
from arcgateway.executor import AsyncioExecutor
from arcgateway.session import SessionRouter

agent_did = "did:arc:agent:slack-helper"

_agent = ToyAgent(agent_did)


async def slack_factory(agent_did_arg: str):
    return _agent


executor = AsyncioExecutor(agent_factory=slack_factory)


async def end_to_end():
    platform = FakeSlackPlatform()
    router = SessionRouter(executor=executor)
    # The router enforces a pairing allowlist by default. Approve the user up front.
    router.add_approved_user("did:arc:user:slack:U_ALICE")
    adapter = SyntheticSlackAdapter(
        platform,
        agent_did=agent_did,
        allowed_user_ids={"U_ALICE"},
        on_message=router.handle,
    )
    router.set_adapter(adapter)

    await adapter.connect()
    await platform.push_user_message(user_id="U_ALICE", channel="C_GENERAL", text="hi from slack")
    await platform.push_user_message(user_id="U_ALICE", channel="C_GENERAL", text="another one")
    # Give the router time to drain the queue.
    await asyncio.sleep(0.2)
    await adapter.disconnect()
    return platform.outbox


outbox = await end_to_end()
for msg in outbox:
    print(msg)

{'id': 'msg_100', 'channel': 'C_GENERAL', 'text': 'Heard 1 message(s) on 4f2dfbd9'}
{'id': 'msg_101', 'channel': 'C_GENERAL', 'text': 'Heard 2 message(s) on 4f2dfbd9'}


Each user message returned through `platform.post_message` — the synthetic Slack equivalent
of `chat.postMessage`. The full round-trip — inbound parse → router → executor → agent →
delta stream → adapter `send` — happened without a single platform-specific line outside
the adapter itself.


### 7.4 Outbound: adapters own the wire

Adapters get used for two outbound paths.

**1. Streaming responses** — `StreamBridge` calls `adapter.send()` for each completed turn.

**2. Scheduled / proactive delivery** — driven agent-side: platform modules consume
scheduler bus events and send through their bots. Either way, the adapter's `send()`
is the single platform-shaped exit from the system.

In [22]:
from arcgateway.delivery import DeliveryTarget


async def proactive_demo():
    platform = FakeSlackPlatform()
    adapter = SyntheticSlackAdapter(
        platform,
        agent_did=agent_did,
        allowed_user_ids={"U_ALICE"},
        on_message=lambda _e: asyncio.sleep(0),  # we're not testing inbound here
    )

    # Could be a string from a TOML file or a parsed DeliveryTarget.
    await adapter.send(DeliveryTarget.parse("slack:C_ALERTS"), "deploy succeeded")
    await adapter.send(DeliveryTarget(platform="slack", chat_id="C_ALERTS"), "second alert")
    return platform.outbox


for entry in await proactive_demo():
    print(entry)

{'id': 'msg_100', 'channel': 'C_ALERTS', 'text': 'deploy succeeded'}
{'id': 'msg_101', 'channel': 'C_ALERTS', 'text': 'second alert'}


**Unknown platform → fail closed.** The sender never silently drops; it raises so the
caller (CronRunner, scheduler) can log and retry.


In [23]:
target = DeliveryTarget.parse("discord:C_X")  # unknown platform: logged, not raised
print(target)

discord:C_X


## 8. Subprocess + NATS executors — design intent and tier fit

The adapter never picks an executor — `GatewayRunner.from_config` does, based on the
configured tier. But the adapter author should know what's behind the curtain because the
executor choice changes failure semantics.

Three executors all satisfy the same `Executor` Protocol:

| Executor | Tier | Isolation | Notes |
|---|---|---|---|
| `AsyncioExecutor` | personal / enterprise | shared event loop | fast, zero startup cost, no resource limits |
| `SubprocessExecutor` | federal | OS process | resource limits via `RLIMIT_*`, tamper-resistant |
| `NATSExecutor` | multi-instance | network | deferred (SPEC-018), no ETA |

The adapter doesn't care which one is wired — but failure modes propagate differently:
an `AsyncioExecutor` agent crash bubbles up as a Python exception caught inside the
executor; a `SubprocessExecutor` worker OOM is detected via exit code; a NATS executor
loses the worker connection entirely. Same `Delta` stream interface, very different blast
radius.


In [24]:
from arcgateway.executor import AsyncioExecutor, Executor, ResourceLimits, SubprocessExecutor
from arcgateway.executor import NATSExecutor

asyncio_exec = AsyncioExecutor()
subprocess_exec = SubprocessExecutor()  # default ResourceLimits
nats_exec = NATSExecutor()

for ex in (asyncio_exec, subprocess_exec, nats_exec):
    print(f"{type(ex).__name__:<22} satisfies Executor: {isinstance(ex, Executor)}")

AsyncioExecutor        satisfies Executor: True
SubprocessExecutor     satisfies Executor: True
NATSExecutor           satisfies Executor: True


**`SubprocessExecutor` is implemented today (T1.6).** It spawns `arc-agent-worker` as a
child process per session, applies `RLIMIT_AS` (memory), `RLIMIT_CPU` (cpu seconds), and
`RLIMIT_NOFILE` (file descriptors), and streams JSON-line deltas back over the worker's
stdout. The defaults (512 MB / 60 s / 256 fds) are deliberately tight — operators raise
them per deployment.


In [25]:
limits = ResourceLimits()
print("Default federal-tier limits:")
print(f"  memory_mb        = {limits.memory_mb}")
print(f"  cpu_seconds      = {limits.cpu_seconds}")
print(f"  file_descriptors = {limits.file_descriptors}")

tight = SubprocessExecutor(
    worker_cmd=["arc-agent-worker"],
    resource_limits=ResourceLimits(memory_mb=256, cpu_seconds=30, file_descriptors=128),
)
print("\ntightened limits attached:", tight._resource_limits.memory_mb, "MB")

Default federal-tier limits:
  memory_mb        = 512
  cpu_seconds      = 60
  file_descriptors = 256

tightened limits attached: 256 MB


**`NATSExecutor` is deferred.** It exists in the type system so callers can write
`Executor`-typed code today, but `run()` raises `NotImplementedError` with a pointer to
SPEC-018. Multi-instance scaling — sharing one bot token across replicas behind a load
balancer — is the use case it'll cover when the open question in SDD §6 lands.


In [26]:
async def nats_stub():
    event = InboundEvent(
        platform="slack",
        chat_id="C_X",
        user_did="did:arc:user:x",
        agent_did="did:arc:agent:y",
        session_key="k",
        message="m",
    )
    try:
        await nats_exec.run(event)
    except NotImplementedError as exc:
        return str(exc)


print(await nats_stub())

NATSExecutor: multi-instance NATS-based scaling is deferred. No implementation ETA in SPEC-018. See SDD §6.


**Adapter authors don't need to do anything different per executor** — the `Executor`
Protocol guarantees the same `Delta` stream surface. But two operational facts matter:

- *In-process executors share the adapter's failure domain.* An adapter bug that monopolises
  the event loop will starve in-process agent runs. With `SubprocessExecutor`, the worker
  has its own loop.
- *Subprocess executors mean serialisation overhead per turn.* If your platform's expected
  latency is dominated by JSON-encoding the inbound event, you'll see it.


## 9. Failure modes — what's likely to bite, and what handles it for you

Adapters are the part of the system that talks to the outside world, so they're the part
most likely to fail in a way that needs explicit handling. Five families to know.


### 9.1 Adapter `connect()` raises

Bad credentials, invalid bot token, dead remote service. The runner catches the exception,
marks the adapter `FAILED`, and adds it to `_failed_adapters` for the reconnect watcher to
retry with exponential backoff. Sibling adapters are unaffected — TaskGroup isolation.


In [27]:
from arcgateway.adapters._reconnect import FailedAdapter

fa = FailedAdapter(name="slack", attempt=0, last_error=RuntimeError("bad token"))
print("Backoff sequence (seconds) for first 6 attempts:")
for i in range(1, 7):
    fa.attempt = i
    print(f"  attempt {i}: {fa.next_backoff_seconds():.0f}s")
print("After 20 attempts: marked PERMANENTLY_FAILED, FATAL audit event emitted.")

Backoff sequence (seconds) for first 6 attempts:
  attempt 1: 30s
  attempt 2: 60s
  attempt 3: 120s
  attempt 4: 240s
  attempt 5: 300s
  attempt 6: 300s
After 20 attempts: marked PERMANENTLY_FAILED, FATAL audit event emitted.


### 9.2 Adapter `disconnect()` raises

It MUST NOT. The contract is: log, swallow, return. We saw the pattern in the synthetic
adapter — `try: await self._poll_task except (CancelledError, Exception): pass`. If your
adapter raises here, gateway shutdown leaks tasks.


### 9.3 Inbound event arrives malformed

If you build an `InboundEvent` with bad fields, Pydantic raises at construction. **Catch it
in the adapter** — never let a malformed event reach the router.


In [28]:
from pydantic import ValidationError

try:
    InboundEvent(
        platform="slack",
        chat_id="C_X",
        # missing user_did / agent_did
        session_key="k",
        message="m",
    )
except ValidationError as exc:
    fields = [e["loc"][0] for e in exc.errors()]
    print("missing required fields:", fields)

missing required fields: ['user_did', 'agent_did']


### 9.4 Executor crashes mid-stream

`AsyncioExecutor` catches the exception, yields a `[agent-error]` token, and emits the
`done` sentinel — we saw this above with `FlakyAgent`. The adapter receives a fully-formed
delta stream and can deliver the error message to the user.


### 9.5 Delivery fails

`adapter.send()` raises (rate limit, network blip). Callers — `StreamBridge`,
the agent-side platform modules — catch, log, and move on. Failures are
audited, never silently dropped.

In [29]:
class BrokenAdapter:
    name = "slack"

    async def connect(self):
        pass

    async def disconnect(self):
        pass

    async def send(self, target, message, *, reply_to=None):
        raise RuntimeError("slack 429 rate limited")

    async def send_with_id(self, target, message):
        await self.send(target, message)
        return None


async def delivery_failure_demo():
    adapter = BrokenAdapter()
    try:
        await adapter.send("slack:C_X", "hi")
    except RuntimeError as exc:
        return f"caught at caller: {exc}"


print(await delivery_failure_demo())

caught at caller: slack 429 rate limited


### 9.6 Wrong-platform delivery

If a `DeliveryTarget` reaches an adapter whose platform doesn't match its
own `name`, the adapter raises — it shouldn't trust the router to have
routed correctly.

## 10. Summary

**The adapter is the gateway's only platform-shaped surface.** Everything inside speaks
Arc primitives. That decision lets the gateway support a new chat platform with one new
file plus the existing `BasePlatformAdapter` Protocol — and lets you test the entire
gateway against synthetic adapters with no live wire formats.

What we covered:

- **`InboundEvent`** — the normalised envelope every adapter must produce. Platform
  locator, identity, payload + raw blob.
- **`DeliveryTarget`** — colon-delimited `platform:chat_id[:thread_id]`. Survives TOML, CLI,
  JSON. Round-trips. Lower-cases platform. Rejects empty `chat_id`. Logs (doesn't raise) on
  unknown platforms.
- **`Delta`** — `kind ∈ {token, tool_call, done}`, `is_final` true only on the final delta.
  `turn_id` keys idempotency.
- **`Executor` Protocol** — `async def run(event) -> AsyncIterator[Delta]`. Setup raises
  from `run`; streaming yields from the iterator. *Always* end with the `done` sentinel.
- **`AsyncioExecutor`** — in-process, shared loop, default for personal / enterprise.
  Wraps an `agent_factory` async callable; falls back to an echo stub when none is wired.
  Catches agent errors and converts them into deltas.
- **`set_agent_factory` / `agent_factory`** — composable wrapping point used by arcui's
  embedded-agent hooks.
- **`SubprocessExecutor` + `ResourceLimits`** — federal-tier process isolation with
  `RLIMIT_AS / RLIMIT_CPU / RLIMIT_NOFILE`. Same Protocol, different blast radius.
- **`NATSExecutor`** — multi-instance scaling, deferred (SPEC-018, no ETA).
- **`BasePlatformAdapter` Protocol** — `name`, `connect`, `disconnect`, `send`,
  `send_with_id`. Four async methods plus a string identifier.
  delivery. `register_adapter(platform, adapter)`, then `send(target, message)` with
  either a `DeliveryTarget` or a raw colon-delimited string.
- **`FailedAdapter` + reconnect watcher** — exponential backoff `30 * 2**(n-1)` capped at
  300s, max 20 attempts before `PERMANENTLY_FAILED`.
- **Synthetic Slack-like adapter** — full lifecycle in ~80 lines: connect, poll, normalise,
  filter, hand off to router, send back, disconnect cleanly.

### Public API exercised

From `arcgateway`: `InboundEvent`, `Delta`, `DeliveryTarget`, `Executor`, `AsyncioExecutor`,
`build_session_key`, `SessionRouter`, `GatewayRunner`.

From `arcgateway.adapters.base`: `BasePlatformAdapter`.
From `arcgateway.adapters._reconnect`: `FailedAdapter`.

From `arcgateway.executor`: `SubprocessExecutor`, `ResourceLimits`, `NATSExecutor`.


### Where to next

- See `walkthroughs/arcgateway/01-session-routing.ipynb` for the `SessionRouter` race-guard
  and `GatewayRunner` lifecycle work this notebook brushed past.
- See `packages/arcgateway/src/arcgateway/adapters/{telegram,slack,mattermost,web}.py` for
  production adapter implementations against real platforms — the same shape as the
  synthetic one above, with platform-specific edges (rate-limit handling, dedup tables,
  reconnect logic, format-aware splitting).
- See `walkthroughs/arcui/01-dashboard-bringup.ipynb` for how arcui reads gateway/agent
  activity back out — arcui is a pure reader of the durable arcstore record (SPEC-026),
  not a live push subscriber to the adapter layer.

### Backlog (referenced from CHANGELOG / SDD)

- **T1.6 — `SubprocessExecutor` end-to-end integration** — implemented and unit-tested;
  full subprocess round-trip integration tests live in
  `tests/integration/test_subprocess_executor.py`.
- **T1.7 — additional platform adapters** — Telegram / Slack / Mattermost / Web shipped;
  Discord, WhatsApp, Signal, Matrix, Email still pending. The Protocol is what new ones
  implement; the synthetic adapter above is the template.
- **`NATSExecutor`** — deferred per SDD §6 / SPEC-018 open question on multi-instance
  scaling. The Protocol slot is reserved.
